# ppo_alpha05_start1

Current 100k/100k PPO, alpha=.5, retaining the complete observed-prefix CNN; change only training episode initialization to always t=1.

Three fresh seeds (0/1/2), exact same 800/200 split, base WAIT coefficient 0.0001. Actual Python notebook cells are executed in process and preserve their outputs and timings, matching the preceding nine experiments. No Jupyter kernel is used. Validation and TEST are both evaluated after checkpoints are locked. Exactly 15 top-level items are delivered.

In [1]:
from pathlib import Path
import sys, os, subprocess, json, hashlib, datetime, time, tempfile
NAME='ppo_alpha05_start1'
PROJECT=Path.cwd()
SEEDS=[0,1,2]
THREADS=2
RUN_TRAINING=True
RUN_FINAL_TEST=True
INSTALL_REQUIREMENTS=False
TRAINING_OUTPUT=PROJECT/'models'
DATA=PROJECT/'data'
print('Actual execution', NAME, datetime.datetime.now(datetime.timezone.utc).isoformat(),flush=True)
print('Python',sys.executable,flush=True)


Actual execution ppo_alpha05_start1 2026-11-17T13:41:34.780779+00:00
Python /workspace/scratch/841abc45f046/.venv_alpha0/bin/python


In [2]:
if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable,'-m','pip','install','torch==2.5.1+cpu',
        '--extra-index-url','https://download.pytorch.org/whl/cpu',
        'stable-baselines3==2.7.0','sb3-contrib==2.7.0','gymnasium==1.2.2',
        'numpy==2.3.5','scipy==1.17.0','scikit-learn==1.8.0','cloudpickle==3.1.2'],check=True)
import platform,torch,numpy,scipy,sklearn,stable_baselines3,sb3_contrib,gymnasium,cloudpickle
versions=dict(python=platform.python_version(),torch=torch.__version__,numpy=numpy.__version__,
    scipy=scipy.__version__,sklearn=sklearn.__version__,sb3=stable_baselines3.__version__,
    sb3_contrib=sb3_contrib.__version__,gymnasium=gymnasium.__version__,cloudpickle=cloudpickle.__version__)
expected=dict(python='3.12.14',torch='2.5.1+cpu',numpy='2.3.5',scipy='1.17.0',
    sklearn='1.8.0',sb3='2.7.0',sb3_contrib='2.7.0',gymnasium='1.2.2',cloudpickle='3.1.2')
assert versions==expected,(versions,expected)
for seed in SEEDS:
    if RUN_TRAINING:
        assert not (TRAINING_OUTPUT/f'seed{seed}'/'config.json').exists(),'Choose a new empty PROJECT for fresh training'
print('Same pinned nine-experiment reference runtime verified:',json.dumps(versions))


Same pinned nine-experiment reference runtime verified: {"python": "3.12.14", "torch": "2.5.1+cpu", "numpy": "2.3.5", "scipy": "1.17.0", "sklearn": "1.8.0", "sb3": "2.7.0", "sb3_contrib": "2.7.0", "gymnasium": "1.2.2", "cloudpickle": "3.1.2"}


In [3]:
(PROJECT/'ppo_wafer.py').write_text('"""On-policy PPO for the Martinez early-classification formulation on UCR2015.\n\nOnly genuine observed prefixes enter the policy. No world model, classifier\npretraining, auxiliary classification loss, label input, or explicit clock input.\nThe original data bytes are checked. Training never opens the official TEST.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport random\nimport time\nfrom pathlib import Path\n\nimport gymnasium as gym\nimport numpy as np\nimport torch\nfrom gymnasium import spaces\nfrom torch import nn\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score\nfrom sklearn.model_selection import train_test_split\nfrom sb3_contrib import MaskablePPO\nfrom sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy\nfrom stable_baselines3.common.callbacks import BaseCallback\nfrom stable_baselines3.common.torch_layers import BaseFeaturesExtractor\nfrom stable_baselines3.common.vec_env import DummyVecEnv\nfrom balanced_ppo import BalancedMaskablePPO, BalancedRolloutBuffer, inverse_frequency_weights\n\nWAIT, CLASS0, CLASS1 = 0, 1, 2\nHASHES = {\n    \'TRAIN\': \'31bba0c41bd068e5b9f3fc04ce7856dc23d93758d2e09cc53ca4a32d3b7a0f05\',\n    \'TEST\': \'bfce80cbcb14d2623c64e70c81ef8a83204c79a5fa739cd247c78b5b34610605\',\n}\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, allow_nan=False), encoding=\'utf-8\')\n\n\ndef load_split(directory, split):\n    path = Path(directory) / f\'wafer_{split}\'\n    if hashlib.sha256(path.read_bytes()).hexdigest() != HASHES[split]:\n        raise ValueError(f\'{path} does not match the verified original UCR2015 bytes\')\n    a = np.loadtxt(path, delimiter=\',\', dtype=np.float32)\n    assert a.shape == ((1000 if split == \'TRAIN\' else 6164), 153)\n    assert np.isfinite(a).all() and set(np.unique(a[:, 0])) == {-1.0, 1.0}\n    return a[:, 1:].copy(), (a[:, 0] == 1).astype(np.int64)\n\n\nclass PrefixWaferEnv(gym.Env):\n    """Self-contained observed prefix, right-padded with zeros to length T.\n\n    Padding is a representation of the variable-length prefix, not an extra\n    clock feature. Labels and the unobserved suffix are absent from observations.\n    Classification ends the episode. At T only the two class actions are legal.\n    Masks are applied inside PPO, so the logged sampling probabilities remain\n    consistent with the actions actually executed.\n    """\n    metadata = {\'render_modes\': []}\n\n    def __init__(self, X, y, seed=0, wait_lambda=0.001, wait_power=1/3,\n                 balanced=True, random_start_fraction=0.5,\n                 correct_reward=1., incorrect_reward=-1., reward_class_weights=(1., 1.),\n                 sampler=\'standard\'):\n        super().__init__()\n        self.X, self.y = np.asarray(X, np.float32), np.asarray(y, np.int64)\n        self.T = self.X.shape[1]\n        self.wait_lambda, self.wait_power = float(wait_lambda), float(wait_power)\n        self.balanced, self.random_start_fraction = balanced, random_start_fraction\n        self.correct_reward, self.incorrect_reward = float(correct_reward), float(incorrect_reward)\n        self.reward_class_weights = np.asarray(reward_class_weights, np.float32)\n        self.sampler, self.training_progress = sampler, 0.\n        self.natural_probabilities = np.bincount(self.y,minlength=2)/len(self.y)\n        self.by_class = [np.flatnonzero(self.y == c) for c in (0, 1)]\n        self.action_space = spaces.Discrete(3)\n        self.observation_space = spaces.Box(-np.inf, np.inf, (self.T,), np.float32)\n        self.rng = np.random.default_rng(seed)\n        self.index, self.t, self.done = 0, 1, True\n\n    def set_training_progress(self, fraction):\n        self.training_progress = float(np.clip(fraction,0,1))\n\n    def sampling_probabilities(self):\n        if self.sampler==\'class_anneal\':\n            minority=int(np.argmin(self.natural_probabilities))\n            fraction=min(self.training_progress/.5,1.)\n            p_minority=(1-fraction)*.5+fraction*self.natural_probabilities[minority]\n            return np.array([p_minority,1-p_minority]) if minority==0 else np.array([1-p_minority,p_minority])\n        return np.array([.5,.5]) if self.balanced else self.natural_probabilities.copy()\n\n    def curriculum_lower_bound(self):\n        if self.sampler!=\'prefix_curriculum\' or self.training_progress>=.6:\n            return None\n        initial=int(np.ceil(.75*self.T))\n        fraction=np.clip((self.training_progress-.2)/.4,0,1)\n        return int(np.ceil(1+(initial-1)*(1-fraction)))\n\n    def observation(self):\n        obs = np.zeros(self.T, np.float32)\n        obs[:self.t] = self.X[self.index, :self.t]\n        return obs\n\n    def reset(self, *, seed=None, options=None):\n        super().reset(seed=seed)\n        if seed is not None:\n            self.rng = np.random.default_rng(seed)\n        options = options or {}\n        probabilities=self.sampling_probabilities()\n        if \'index\' in options:\n            self.index = int(options[\'index\'])\n        elif self.sampler==\'class_anneal\':\n            label=0 if self.rng.random()<probabilities[0] else 1\n            self.index=int(self.rng.choice(self.by_class[label]))\n        elif self.balanced:\n            self.index = int(self.rng.choice(self.by_class[int(self.rng.integers(2))]))\n        else:\n            self.index = int(self.rng.integers(len(self.y)))\n        if \'start_at\' in options:\n            self.t = int(options[\'start_at\'])\n        else:\n            lower=self.curriculum_lower_bound()\n            self.t = (int(self.rng.integers(lower,self.T+1)) if lower is not None else\n                (int(self.rng.integers(1, self.T + 1))\n                 if self.rng.random() < self.random_start_fraction else 1))\n        assert 1 <= self.t <= self.T\n        self.done = False\n        self.episode_class_probability=float(probabilities[self.y[self.index]])\n        self.episode_start_t=self.t\n        return self.observation(), {}\n\n    def action_masks(self):\n        return np.array([self.t < self.T, True, True], dtype=bool)\n\n    def step(self, action):\n        action = int(action)\n        if self.done:\n            raise RuntimeError(\'Reset required after an episode\')\n        if action == WAIT:\n            if self.t >= self.T:\n                raise ValueError(\'WAIT is masked at the final measurement\')\n            reward = -self.wait_lambda * self.t ** self.wait_power * self.reward_class_weights[self.y[self.index]]\n            info = {\'label\': int(self.y[self.index]), \'series_index\': self.index,\n                    \'observation_time\': self.t,\n                    \'episode_class_probability\': self.episode_class_probability,\n                    \'episode_start_t\': self.episode_start_t}\n            self.t += 1\n            return self.observation(), float(reward), False, False, info\n        if action not in (CLASS0, CLASS1):\n            raise ValueError(action)\n        pred = action - 1\n        correct = pred == self.y[self.index]\n        reward = (self.correct_reward if correct else self.incorrect_reward) * self.reward_class_weights[self.y[self.index]]\n        self.done = True\n        return self.observation(), reward, True, False, {\n            \'prediction\': pred, \'label\': int(self.y[self.index]),\n            \'decision_time\': self.t, \'correct\': bool(correct),\n            \'series_index\': self.index, \'observation_time\': self.t,\n            \'episode_class_probability\': self.episode_class_probability,\n            \'episode_start_t\': self.episode_start_t,\n        }\n\n\nclass PrefixCNN(BaseFeaturesExtractor):\n    """Small 1-D CNN over the observed prefix, with no normalization layers."""\n    def __init__(self, observation_space, features_dim=128):\n        super().__init__(observation_space, features_dim)\n        self.conv = nn.Sequential(\n            nn.Conv1d(1, 8, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),\n            nn.Conv1d(8, 16, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),\n            nn.Flatten(),\n        )\n        with torch.no_grad():\n            size = self.conv(torch.zeros(1, 1, observation_space.shape[0])).shape[1]\n        self.linear = nn.Sequential(nn.Linear(size, features_dim), nn.ReLU())\n\n    def forward(self, observations):\n        return self.linear(self.conv(observations.unsqueeze(1)))\n\n\nclass ExploringPolicy(MaskableActorCriticPolicy):\n    """A declared epsilon-uniform policy family with exact on-policy log-probs.\n\n    Mixing happens before sampling, in every forward/evaluate_actions path.\n    The legal-action mask is then applied consistently by MaskablePPO.\n    This is not an external action override or off-policy replay adaptation.\n    """\n    def __init__(self, *args, exploration_mix=0.1, **kwargs):\n        self.exploration_mix = float(exploration_mix)\n        super().__init__(*args, **kwargs)\n\n    def _get_action_dist_from_latent(self, latent_pi):\n        logits = self.action_net(latent_pi)\n        probs = logits.softmax(-1)\n        mixed = (1 - self.exploration_mix) * probs + self.exploration_mix / probs.shape[-1]\n        return self.action_dist.proba_distribution(action_logits=mixed.log())\n\n\n@torch.no_grad()\ndef evaluate(model, X, y, wait_lambda=0.001, wait_power=1/3,\n             stochastic=False, seed=0, return_predictions=False,\n             correct_reward=1., incorrect_reward=-1., reward_class_weights=(1., 1.)):\n    """Every evaluation starts from t=1, with natural data frequencies."""\n    old_mode = model.policy.training\n    model.policy.set_training_mode(False)\n    n, T = X.shape\n    pred = np.full(n, -1, np.int64)\n    stops = np.full(n, T, np.int64)\n    first_probs = np.zeros((n, 3), np.float32)\n    action_counts = np.zeros(3, np.int64)\n    with torch.random.fork_rng(devices=[]):\n        torch.manual_seed(seed)\n        for t in range(1, T + 1):\n            active = np.flatnonzero(pred < 0)\n            if not len(active):\n                break\n            if model.observation_space.shape == (1,):\n                obs = X[active, t-1:t].copy()\n            else:\n                obs = np.zeros((len(active), T), np.float32)\n                obs[:, :t] = X[active, :t]\n            masks = np.ones((len(active), 3), bool)\n            masks[:, WAIT] = t < T\n            tensors = torch.as_tensor(obs, device=model.device)\n            dist = model.policy.get_distribution(tensors, action_masks=masks)\n            probs = dist.distribution.probs\n            actions = (torch.multinomial(probs, 1).squeeze(-1) if stochastic\n                       else probs.argmax(-1)).cpu().numpy()\n            if t == 1:\n                first_probs[active] = probs.cpu().numpy()\n            action_counts += np.bincount(actions, minlength=3)\n            chosen = actions != WAIT\n            pred[active[chosen]] = actions[chosen] - 1\n            stops[active[chosen]] = t\n    model.policy.set_training_mode(old_mode)\n    assert (pred >= 0).all()\n    cm = confusion_matrix(y, pred, labels=[0, 1])\n    costs = np.r_[0., np.cumsum(wait_lambda * np.arange(1, T) ** wait_power)]\n    ret = (np.where(pred == y, correct_reward, incorrect_reward)-costs[stops-1]) * np.asarray(reward_class_weights)[y]\n    result = dict(\n        n=int(n), evaluation=\'sample\' if stochastic else \'greedy\',\n        accuracy=float(accuracy_score(y, pred)),\n        balanced_accuracy=float(balanced_accuracy_score(y, pred)),\n        macro_f1=float(f1_score(y, pred, average=\'macro\', zero_division=0)),\n        confusion_matrix=cm.tolist(),\n        recall_by_class=[float(cm[c, c] / cm[c].sum()) for c in (0, 1)],\n        avg_t=float(stops.mean()), earliness=float(stops.mean() / T),\n        first_stop_fraction=float((stops == 1).mean()), mean_return=float(ret.mean()),\n        avg_t_by_class=[float(stops[y == c].mean()) for c in (0, 1)],\n        action_counts=action_counts.tolist(),\n        first_probs_by_class=[first_probs[y == c].mean(0).tolist() for c in (0, 1)],\n    )\n    if return_predictions:\n        return result, dict(labels=y, predictions=pred, decision_times=stops)\n    return result\n\n\nclass EvaluateCallback(BaseCallback):\n    def __init__(self, X, y, output, args):\n        super().__init__()\n        self.X, self.y, self.output, self.args = X, y, Path(output), args\n        self.best_key = (-1., -float(\'inf\'))\n        self.next_eval = args.eval_every\n        self.started = time.monotonic()\n        self.best_early_key=(-1.,-float(\'inf\'))\n        self.saved_reference=False\n        self.actual_gradient_steps=0\n\n    def _on_training_start(self):\n        def count_update(optimizer,args,kwargs):\n            self.actual_gradient_steps+=1\n        self.optimizer_hook=self.model.policy.optimizer.register_step_post_hook(count_update)\n\n    def _on_rollout_start(self):\n        self.rollout_labels, self.rollout_actions = [], []\n        self.rollout_probabilities,self.rollout_starts=[],[]\n        if self.args.sampler!=\'standard\':\n            self.training_env.env_method(\'set_training_progress\',self.num_timesteps/self.args.steps)\n        if self.args.save_reference_at and self.num_timesteps==self.args.save_reference_at and not self.saved_reference:\n            self.model.save(self.output/\'reference_1m_model\')\n            self.saved_reference=True\n\n    def _on_rollout_end(self):\n        labels = np.asarray(self.rollout_labels)\n        actions = np.asarray(self.rollout_actions)\n        buffer = self.model.rollout_buffer\n        weights = (buffer.class_weights[labels] if isinstance(buffer, BalancedRolloutBuffer)\n                   else np.ones_like(labels, dtype=np.float32))\n        if isinstance(buffer,BalancedRolloutBuffer) and buffer.per_transition_weights:\n            weights=buffer.transition_weights[:,:,0]\n        counts = np.bincount(labels.flatten(), minlength=2)\n        row = dict(env_steps=int(self.num_timesteps), transition_class_counts=counts.tolist(),\n            sampler=self.args.sampler,actual_gradient_steps=self.actual_gradient_steps,\n            mean_episode_start=float(np.mean(self.rollout_starts)),\n            mean_episode_class_probability=float(np.mean(self.rollout_probabilities)),\n            weighted_class_mass=[float(weights[labels==c].sum()/weights.sum()) for c in (0,1)],\n            action_counts_by_class=[np.bincount(actions[labels==c],minlength=3).tolist() for c in (0,1)],\n            mean_advantage_by_class=[float(buffer.advantages[labels==c].mean()) if counts[c] else None for c in (0,1)],\n            mean_return_by_class=[float(buffer.returns[labels==c].mean()) if counts[c] else None for c in (0,1)])\n        with (self.output/\'rollout_diagnostics.jsonl\').open(\'a\') as f:\n            f.write(json.dumps(row)+\'\\n\')\n\n    def perform(self, final=False):\n        a = self.args\n        result = evaluate(self.model, self.X, self.y, a.wait_lambda, a.wait_power,\n            correct_reward=a.correct_reward, incorrect_reward=a.incorrect_reward,\n            reward_class_weights=a.reward_class_weights)\n        result.update(env_steps=int(self.num_timesteps), ppo_epochs=int(self.model._n_updates),\n                      actual_gradient_steps=self.actual_gradient_steps,\n                      seconds=time.monotonic() - self.started, final=final)\n        with (self.output / \'monitor.jsonl\').open(\'a\') as f:\n            f.write(json.dumps(result) + \'\\n\')\n        key = (result[\'accuracy\'], result[\'mean_return\'])\n        if key > self.best_key:\n            self.best_key = key\n            self.model.save(self.output / \'best_model\')\n            write_json(self.output / \'best_monitor.json\', result)\n        if self.args.max_validation_mean_t and result[\'avg_t\']<=self.args.max_validation_mean_t and key>self.best_early_key:\n            self.best_early_key=key\n            self.model.save(self.output/\'early_best_model\')\n            write_json(self.output/\'early_best_monitor.json\',result)\n        self.model.save(self.output / \'last_model\')\n        print(f"[eval] step={self.num_timesteps:,} acc={result[\'accuracy\']:.5f} "\n              f"BA={result[\'balanced_accuracy\']:.5f} t={result[\'avg_t\']:.3f} "\n              f"return={result[\'mean_return\']:.4f} elapsed={result[\'seconds\']:.1f}s "\n              f"best_acc={self.best_key[0]:.5f}", flush=True)\n        return result\n\n    def _on_step(self):\n        self.rollout_labels.append([info[\'label\'] for info in self.locals[\'infos\']])\n        self.rollout_actions.append(np.asarray(self.locals[\'actions\']).reshape(-1).tolist())\n        self.rollout_probabilities.append([info[\'episode_class_probability\'] for info in self.locals[\'infos\']])\n        self.rollout_starts.append([info[\'episode_start_t\'] for info in self.locals[\'infos\']])\n        buffer = self.locals.get(\'rollout_buffer\')\n        if isinstance(buffer, BalancedRolloutBuffer):\n            labels = np.array([info[\'label\'] for info in self.locals[\'infos\']], dtype=np.int64)\n            buffer.class_labels[buffer.pos, :, 0] = labels\n            if buffer.per_transition_weights:\n                probability=np.asarray(self.rollout_probabilities[-1],dtype=np.float64)\n                alpha=self.args.actor_balance_alpha\n                raw=(1/(2*probability))**alpha\n                normalizer=probability*raw+(1-probability)*(1/(2*(1-probability)))**alpha\n                buffer.transition_weights[buffer.pos,:,0]=(raw/normalizer).astype(np.float32)\n        if self.num_timesteps >= self.next_eval:\n            self.perform()\n            self.next_eval += self.args.eval_every\n        return True\n\n    def _on_training_end(self):\n        self.perform(final=True)\n        write_json(self.output/\'training_summary.json\',dict(completed=True,\n            env_steps=int(self.num_timesteps),actual_gradient_steps=self.actual_gradient_steps,\n            model_parameters=sum(p.numel() for p in self.model.policy.parameters())))\n        self.optimizer_hook.remove()\n\n\ndef check_environment(X, y):\n    """Meaningful task checks before a potentially long training run."""\n    env = PrefixWaferEnv(X, y, seed=0)\n    ix = int(np.flatnonzero(y == 0)[0])\n    obs, _ = env.reset(options=dict(index=ix, start_at=4))\n    altered = X.copy(); altered[:, 4:] = 1e6\n    other = PrefixWaferEnv(altered, y)\n    obs2, _ = other.reset(options=dict(index=ix, start_at=4))\n    assert np.array_equal(obs, obs2), \'Unobserved suffix leaked into policy inputs\'\n    assert np.array_equal(obs[:4], X[ix, :4]) and np.count_nonzero(obs[4:]) == 0\n    after, reward, done, trunc, _ = env.step(WAIT)\n    assert np.isclose(reward, -.001 * 4 ** (1/3)) and not done and not trunc\n    assert np.array_equal(after[:5], X[ix, :5]) and np.count_nonzero(after[5:]) == 0\n    for label in (0, 1):\n        ix = int(np.flatnonzero(y == label)[0])\n        for action in (CLASS0, CLASS1):\n            env.reset(options=dict(index=ix, start_at=16))\n            _, r, done, _, info = env.step(action)\n            assert r == (1. if action - 1 == label else -1.) and done\n            assert info[\'decision_time\'] == 16\n    env.reset(options=dict(index=ix, start_at=X.shape[1]))\n    assert env.action_masks().tolist() == [False, True, True]\n    print(\'[checks] prefix causality, action/reward mapping, WAIT cost and final mask passed\', flush=True)\n\n\ndef train(args):\n    if args.balance_value_loss and not args.actor_balance_alpha:\n        raise ValueError(\'--balance-value-loss requires --actor-balance-alpha > 0\')\n    if args.correct_reward <= args.incorrect_reward or args.wait_lambda < 0:\n        raise ValueError(\'Correct decisions must pay more than incorrect ones; WAIT cost must be nonnegative\')\n    if args.sampler!=\'standard\' and args.reward_balance_alpha:\n        raise ValueError(\'These controlled sampler experiments keep rewards unweighted\')\n    torch.set_num_threads(args.threads)\n    torch.use_deterministic_algorithms(True)\n    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)\n    X, y = load_split(args.data_dir, \'TRAIN\')\n    check_environment(X, y)\n    indices = np.arange(len(y))\n    if args.protocol == \'holdout\':\n        train_ids, monitor_ids = train_test_split(indices, test_size=.2,\n            stratify=y, random_state=args.split_seed)\n        monitor_name = \'independent_validation\'\n    else:\n        train_ids = monitor_ids = indices\n        monitor_name = \'in_sample_train_monitor\'\n    if args.train_order == \'sorted\':\n        train_ids = np.sort(train_ids)\n    output = Path(args.output); output.mkdir(parents=True, exist_ok=True)\n    if (output / \'config.json\').exists():\n        raise FileExistsError(f\'Use a new output directory: {output}\')\n    # Weight the ACTUAL episode distribution; do not correct imbalance twice\n    # when the environment already samples classes equally.\n    sampling_labels = y[train_ids] if args.natural_sampling else np.array([0,1])\n    actor_weights = inverse_frequency_weights(sampling_labels, args.actor_balance_alpha)\n    args.reward_class_weights = inverse_frequency_weights(sampling_labels, args.reward_balance_alpha).tolist()\n    write_json(output / \'config.json\', vars(args))\n    write_json(output / \'split_manifest.json\', dict(\n        dataset=\'original UCR2015 Wafer\', train_hash=HASHES[\'TRAIN\'],\n        train_indices=train_ids.tolist(), monitor_indices=monitor_ids.tolist(),\n        monitor=monitor_name, test_used_for_selection=False,\n        observation=\'right-zero-padded real prefix; no clock, label, or future values\',\n        algorithm=\'SB3-contrib MaskablePPO, real on-policy rollouts\',\n        classification_pretraining=False, auxiliary_classification_loss=False,\n        final_action_set=\'CLASS0, CLASS1; WAIT invalid at T\',\n        reward=dict(correct=args.correct_reward, incorrect=args.incorrect_reward,\n                    wait_lambda=args.wait_lambda, wait_power=args.wait_power,\n                    class_weights=args.reward_class_weights),\n        actor_class_weights=actor_weights.tolist(), actor_balance_alpha=args.actor_balance_alpha,\n        sampler=args.sampler, actor_weights_follow_actual_episode_probability=args.sampler==\'class_anneal\',\n        versions=dict(python=platform.python_version(), torch=torch.__version__,\n                      sb3=__import__(\'stable_baselines3\').__version__,\n                      sb3_contrib=__import__(\'sb3_contrib\').__version__, gymnasium=gym.__version__),\n    ))\n    def make_env(i):\n        return lambda: PrefixWaferEnv(X[train_ids], y[train_ids],\n            seed=args.seed + i, wait_lambda=args.wait_lambda, wait_power=args.wait_power,\n            balanced=not args.natural_sampling, random_start_fraction=args.random_start_fraction,\n            correct_reward=args.correct_reward, incorrect_reward=args.incorrect_reward,\n            reward_class_weights=args.reward_class_weights,sampler=args.sampler)\n    vec = DummyVecEnv([make_env(i) for i in range(args.n_envs)])\n    policy_kwargs = dict(features_extractor_class=PrefixCNN,\n        features_extractor_kwargs=dict(features_dim=128),\n        net_arch=dict(pi=[64], vf=[64]), activation_fn=nn.Tanh)\n    policy_class = ExploringPolicy if args.exploration_mix else \'MlpPolicy\'\n    if args.exploration_mix:\n        policy_kwargs[\'exploration_mix\'] = args.exploration_mix\n    model_class = BalancedMaskablePPO if args.actor_balance_alpha else MaskablePPO\n    buffer_kwargs = dict(rollout_buffer_class=BalancedRolloutBuffer,\n                         rollout_buffer_kwargs=dict(class_weights=actor_weights,\n                            per_transition_weights=args.sampler==\'class_anneal\'),\n                         balance_value_loss=args.balance_value_loss) if args.actor_balance_alpha else {}\n    model = model_class(policy_class, vec, learning_rate=args.lr,\n        n_steps=args.n_steps, batch_size=args.batch_size, n_epochs=args.epochs,\n        gamma=1.0, gae_lambda=args.gae_lambda, clip_range=.2, ent_coef=args.ent_coef,\n        vf_coef=.5, max_grad_norm=.5, target_kl=.03,\n        policy_kwargs=policy_kwargs, seed=args.seed, device=\'cpu\', verbose=0, **buffer_kwargs)\n    if args.initial_wait_bias:\n        # An explicitly recorded initial policy preference, not forced actions.\n        # PPO samples from and optimizes this exact distribution throughout.\n        with torch.no_grad():\n            model.policy.action_net.bias[WAIT] = args.initial_wait_bias\n    if args.init_checkpoint:\n        checkpoint = Path(args.init_checkpoint)\n        warm = BalancedMaskablePPO.load(checkpoint, device=\'cpu\')\n        model.policy.load_state_dict(warm.policy.state_dict())\n        # A fresh optimizer and fresh on-policy rollouts: this is a documented\n        # second PPO stage, not a continuation of stale rollout-buffer samples.\n        write_json(output/\'warm_start.json\', dict(\n            source=str(checkpoint.resolve()),\n            source_sha256=hashlib.sha256(checkpoint.read_bytes()).hexdigest(),\n            optimizer=\'fresh Adam\', classification_pretraining=False,\n            description=\'PPO-trained weights; newly collected real on-policy data\'))\n        del warm\n    callback = EvaluateCallback(X[monitor_ids], y[monitor_ids], output, args)\n    print(f\'[start] {args.output} | seed={args.seed} train={len(train_ids)} \'\n          f\'monitor={len(monitor_ids)} params={sum(p.numel() for p in model.policy.parameters()):,} \'\n          f\'lambda={args.wait_lambda} steps={args.steps:,}\', flush=True)\n    model.learn(args.steps, callback=callback)\n    vec.close()\n    print(\'[done]\', args.output, flush=True)\n\n\ndef test(args):\n    """Explicit final evaluation, once, after the recipe/checkpoint is locked."""\n    torch.set_num_threads(args.threads)\n    root = Path(args.output)\n    result_path = root / \'official_test_once.json\'\n    if result_path.exists():\n        raise FileExistsError(f\'TEST has already been evaluated: {result_path}\')\n    config = json.loads((root / \'config.json\').read_text())\n    explicit_data = Path(args.data_dir)\n    data_dir = explicit_data if (explicit_data/\'wafer_TEST\').is_file() else Path(config[\'data_dir\'])\n    X, y = load_split(data_dir, \'TEST\')\n    checkpoint_name = args.checkpoint + \'_model.zip\'\n    model_class = BalancedMaskablePPO if config.get(\'actor_balance_alpha\', 0) else MaskablePPO\n    model = model_class.load(root / checkpoint_name, device=\'cpu\')\n    result = dict(checkpoint=checkpoint_name, test_hash=HASHES[\'TEST\'],\n                  checkpoint_sha256=hashlib.sha256((root/checkpoint_name).read_bytes()).hexdigest(),\n                  seed=config[\'seed\'], test_used_for_selection=False,\n                  selected_on=(\'fixed_budget_before_TEST\' if args.checkpoint==\'last\' and config[\'protocol\']==\'all_train\'\n                               else json.loads((root/\'split_manifest.json\').read_text())[\'monitor\']))\n    selection_lock = root / \'selection_lock.json\'\n    if selection_lock.is_file():\n        result[\'selection_lock\'] = json.loads(selection_lock.read_text())\n        expected_hash = result[\'selection_lock\'].get(\'checkpoint_sha256\')\n        if expected_hash and expected_hash != result[\'checkpoint_sha256\']:\n            raise ValueError(\'Checkpoint changed after the pre-test selection lock\')\n    for mode in (\'greedy\', \'sample\'):\n        score, predictions = evaluate(model, X, y, config[\'wait_lambda\'], config[\'wait_power\'],\n            stochastic=mode==\'sample\', seed=config[\'seed\']+10000, return_predictions=True,\n            correct_reward=config.get(\'correct_reward\', 1.), incorrect_reward=config.get(\'incorrect_reward\', -1.),\n            reward_class_weights=config.get(\'reward_class_weights\', (1., 1.)))\n        result[mode] = score\n        np.savez_compressed(root / f\'test_predictions_{mode}.npz\', **predictions)\n    write_json(result_path, result)\n    print(json.dumps(result, indent=2), flush=True)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'mode\', choices=[\'train\', \'test\', \'check\'])\n    p.add_argument(\'--data-dir\', default=\'data\')\n    p.add_argument(\'--output\', default=\'runs/ppo_seed0\')\n    p.add_argument(\'--checkpoint\', choices=[\'best\', \'last\'], default=\'best\',\n                   help=\'Explicit final-test checkpoint; choose before reading TEST results\')\n    p.add_argument(\'--protocol\', choices=[\'holdout\', \'all_train\'], default=\'holdout\')\n    p.add_argument(\'--seed\', type=int, default=0)\n    p.add_argument(\'--split-seed\', type=int, default=0)\n    p.add_argument(\'--train-order\', choices=[\'split\', \'sorted\'], default=\'split\',\n                   help=\'Diagnostic only: retain or sort the same training IDs; same seed is not same samples\')\n    p.add_argument(\'--steps\', type=int, default=262144)\n    p.add_argument(\'--sampler\',choices=[\'standard\',\'class_anneal\',\'prefix_curriculum\'],default=\'standard\')\n    p.add_argument(\'--max-validation-mean-t\',type=float,default=0.,\n                   help=\'Also retain a separately selected checkpoint satisfying this validation latency limit\')\n    p.add_argument(\'--save-reference-at\',type=int,default=0,\n                   help=\'Save a post-update prefix checkpoint for the training-budget control\')\n    p.add_argument(\'--n-envs\', type=int, default=32)\n    p.add_argument(\'--n-steps\', type=int, default=64)\n    p.add_argument(\'--batch-size\', type=int, default=256)\n    p.add_argument(\'--epochs\', type=int, default=5)\n    p.add_argument(\'--gae-lambda\', type=float, default=.95)\n    p.add_argument(\'--lr\', type=float, default=3e-4)\n    p.add_argument(\'--ent-coef\', type=float, default=.02)\n    p.add_argument(\'--initial-wait-bias\', type=float, default=0.)\n    p.add_argument(\'--init-checkpoint\', default=None,\n                   help=\'Optional PPO-weight initialization for a recorded second stage; no supervised pretraining\')\n    p.add_argument(\'--exploration-mix\', type=float, default=0.,\n                   help=\'Optional epsilon-uniform policy family, with consistent PPO log-probabilities\')\n    p.add_argument(\'--wait-lambda\', type=float, default=.001)\n    p.add_argument(\'--wait-power\', type=float, default=1/3)\n    p.add_argument(\'--correct-reward\', type=float, default=1.)\n    p.add_argument(\'--incorrect-reward\', type=float, default=-1.)\n    p.add_argument(\'--actor-balance-alpha\', type=float, default=0.,\n                   help=\'0: original actor; 1: inverse-frequency class weights in actor/entropy loss\')\n    p.add_argument(\'--balance-value-loss\', action=\'store_true\',\n                   help=\'Also class-weight critic squared error; requires a nonzero actor-balance-alpha\')\n    p.add_argument(\'--reward-balance-alpha\', type=float, default=0.,\n                   help=\'Class weights in ALL rewards, including WAIT; separate ablation from actor weighting\')\n    p.add_argument(\'--random-start-fraction\', type=float, default=.5)\n    p.add_argument(\'--natural-sampling\', action=\'store_true\')\n    p.add_argument(\'--threads\', type=int, default=2)\n    p.add_argument(\'--eval-every\', type=int, default=16384)\n    args = p.parse_args()\n    if args.mode == \'train\':\n        train(args)\n    elif args.mode == \'test\':\n        test(args)\n    else:\n        check_environment(*load_split(args.data_dir, \'TRAIN\'))\n\n\nif __name__ == \'__main__\':\n    main()\n',encoding='utf-8')
print('Exact source ready: ppo_wafer.py')

Exact source ready: ppo_wafer.py


In [4]:
(PROJECT/'balanced_ppo.py').write_text('"""Class-weighted PPO actor ablation based on sb3-contrib 2.7.0.\n\nLabels are training metadata, never policy observations. Both WAIT and class\nactions receive the weight of their episode\'s true class. The critic remains\nunweighted to isolate the actor change. Sampling and logged action likelihoods\nare the standard masked on-policy ones; no replay or forced actions are used.\n"""\nfrom typing import NamedTuple\n\nimport numpy as np\nimport torch as th\nimport torch.nn.functional as F\nfrom gymnasium import spaces\nfrom sb3_contrib import MaskablePPO\nfrom sb3_contrib.common.maskable.buffers import MaskableRolloutBuffer\nfrom stable_baselines3.common.utils import explained_variance\n\n\nclass BalancedSamples(NamedTuple):\n    observations: th.Tensor\n    actions: th.Tensor\n    old_values: th.Tensor\n    old_log_prob: th.Tensor\n    advantages: th.Tensor\n    returns: th.Tensor\n    action_masks: th.Tensor\n    actor_weights: th.Tensor\n\n\ndef weighted_mean(values, weights):\n    return (values * weights).sum() / weights.sum().clamp_min(1e-12)\n\n\ndef clipped_actor_loss(advantages, ratio, weights, clip_range):\n    unclipped = advantages * ratio\n    clipped = advantages * th.clamp(ratio, 1-clip_range, 1+clip_range)\n    return -weighted_mean(th.minimum(unclipped, clipped), weights)\n\n\nclass BalancedRolloutBuffer(MaskableRolloutBuffer):\n    def __init__(self, *args, class_weights=(1., 1.), per_transition_weights=False, **kwargs):\n        self.class_weights = np.asarray(class_weights, dtype=np.float32)\n        self.per_transition_weights = bool(per_transition_weights)\n        super().__init__(*args, **kwargs)\n\n    def reset(self):\n        super().reset()\n        self.class_labels = np.full((self.buffer_size, self.n_envs, 1), -1, np.int64)\n        self.transition_weights = np.full((self.buffer_size, self.n_envs, 1), np.nan, np.float32)\n\n    def add(self, *args, **kwargs):\n        if not np.isin(self.class_labels[self.pos], [0, 1]).all():\n            raise RuntimeError(\'Training class metadata must be set before buffer.add\')\n        if self.per_transition_weights and not (self.transition_weights[self.pos]>0).all():\n            raise RuntimeError(\'Actual episode-distribution weights must be set before buffer.add\')\n        return super().add(*args, **kwargs)\n\n    def get(self, batch_size=None):\n        if not self.generator_ready:\n            self.class_labels = self.swap_and_flatten(self.class_labels)\n            self.transition_weights = self.swap_and_flatten(self.transition_weights)\n        yield from super().get(batch_size)\n\n    def _get_samples(self, batch_inds, env=None):\n        base = super()._get_samples(batch_inds, env)\n        labels = self.class_labels[batch_inds].reshape(-1)\n        weights = self.to_torch(self.transition_weights[batch_inds].flatten()\n            if self.per_transition_weights else self.class_weights[labels])\n        return BalancedSamples(*base, weights)\n\n\nclass BalancedMaskablePPO(MaskablePPO):\n    """Same clipped PPO update, with class weights in actor and entropy losses."""\n    def __init__(self, *args, balance_value_loss=False, **kwargs):\n        self.balance_value_loss = bool(balance_value_loss)\n        super().__init__(*args, **kwargs)\n\n    def train(self):\n        self.policy.set_training_mode(True)\n        self._update_learning_rate(self.policy.optimizer)\n        clip_range = self.clip_range(self._current_progress_remaining)\n        if self.clip_range_vf is not None:\n            clip_range_vf = self.clip_range_vf(self._current_progress_remaining)\n        entropy_losses, pg_losses, value_losses, clip_fractions = [], [], [], []\n        continue_training = True\n        for epoch in range(self.n_epochs):\n            approx_kl_divs = []\n            for data in self.rollout_buffer.get(self.batch_size):\n                actions = data.actions\n                if isinstance(self.action_space, spaces.Discrete):\n                    actions = actions.long().flatten()\n                values, log_prob, entropy = self.policy.evaluate_actions(\n                    data.observations, actions, action_masks=data.action_masks)\n                values = values.flatten()\n                advantages = data.advantages\n                if self.normalize_advantage:\n                    # Keep the original SB3 normalization; isolate loss weighting.\n                    advantages = (advantages-advantages.mean())/(advantages.std()+1e-8)\n                ratio = th.exp(log_prob-data.old_log_prob)\n                weights = data.actor_weights\n                policy_loss = clipped_actor_loss(advantages, ratio, weights, clip_range)\n                pg_losses.append(policy_loss.item())\n                clip_fractions.append(weighted_mean(\n                    ((ratio-1).abs()>clip_range).float(), weights).item())\n                values_pred = values if self.clip_range_vf is None else data.old_values + th.clamp(\n                    values-data.old_values, -clip_range_vf, clip_range_vf)\n                # Intentionally unchanged: this ablation weights the actor only.\n                value_loss = (weighted_mean((data.returns-values_pred).square(), weights)\n                              if self.balance_value_loss else F.mse_loss(data.returns, values_pred))\n                value_losses.append(value_loss.item())\n                entropy_loss = -weighted_mean(-log_prob if entropy is None else entropy, weights)\n                entropy_losses.append(entropy_loss.item())\n                loss = policy_loss+self.ent_coef*entropy_loss+self.vf_coef*value_loss\n                with th.no_grad():\n                    log_ratio = log_prob-data.old_log_prob\n                    approx_kl = weighted_mean((th.exp(log_ratio)-1)-log_ratio, weights).item()\n                    approx_kl_divs.append(approx_kl)\n                if self.target_kl is not None and approx_kl > 1.5*self.target_kl:\n                    continue_training = False\n                    break\n                self.policy.optimizer.zero_grad()\n                loss.backward()\n                th.nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)\n                self.policy.optimizer.step()\n            if not continue_training:\n                break\n        self._n_updates += self.n_epochs\n        variance = explained_variance(self.rollout_buffer.values.flatten(),\n                                      self.rollout_buffer.returns.flatten())\n        for name, value in dict(entropy_loss=np.mean(entropy_losses),\n                policy_gradient_loss=np.mean(pg_losses), value_loss=np.mean(value_losses),\n                approx_kl=np.mean(approx_kl_divs), clip_fraction=np.mean(clip_fractions),\n                loss=loss.item(), explained_variance=variance,\n                clip_range=clip_range).items():\n            self.logger.record(\'train/\'+name, value)\n        self.logger.record(\'train/n_updates\', self._n_updates, exclude=\'tensorboard\')\n\n\ndef inverse_frequency_weights(labels, alpha):\n    counts = np.bincount(np.asarray(labels, np.int64), minlength=2)\n    if (counts == 0).any():\n        raise ValueError(\'Both classes are required\')\n    frequency = counts / counts.sum()\n    weights = (1/(2*frequency)) ** float(alpha)\n    # Unit expected weight under the training episode distribution.\n    return weights / np.dot(frequency, weights)\n',encoding='utf-8')
print('Exact source ready: balanced_ppo.py')

Exact source ready: balanced_ppo.py


In [5]:
(PROJECT/'equal_budget_ppo.py').write_text('"""Clipped masked PPO with exact, independently counted interaction/update budgets.\n\nSame loss/network as the previous round; the new experimental factor is a much\nhigher number of epochs per fresh rollout. Old action log probabilities and GAE\ntargets stay fixed during these epochs. No old rollout is replayed after the\nnext collection, no log probabilities are relabelled, and no dummy steps count.\n"""\nfrom __future__ import annotations\n\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nimport torch as th\nimport torch.nn.functional as F\nfrom gymnasium import spaces\nfrom stable_baselines3.common.utils import explained_variance\n\nfrom balanced_ppo import BalancedMaskablePPO, BalancedRolloutBuffer, clipped_actor_loss, weighted_mean\nfrom ppo_wafer import EvaluateCallback, write_json\n\n\nclass UpdateEvaluateCallback(EvaluateCallback):\n    """Validation every 5k real optimizer steps, never selected using TEST."""\n\n    def _on_step(self):\n        # Parent also supplies true-class training metadata to the rollout.\n        # Suppress its interaction-based validation trigger in this experiment.\n        old_next = self.next_eval\n        self.next_eval = math.inf\n        result = super()._on_step()\n        self.next_eval = old_next\n        return result\n\n    def after_optimizer_step(self):\n        assert self.actual_gradient_steps == self.model.actual_optimizer_steps\n        if self.actual_gradient_steps >= self.next_eval:\n            self.perform()\n            self.next_eval += self.args.eval_every\n\n    def perform(self, final=False):\n        result = super().perform(final=final)\n        print(f\'[actual updates] {self.actual_gradient_steps:,}\', flush=True)\n        return result\n\n    def _on_training_end(self):\n        assert self.num_timesteps == self.args.steps\n        assert self.actual_gradient_steps == self.model.actual_optimizer_steps\n        if self.args.target_kl is None:\n            assert self.actual_gradient_steps == self.args.optimizer_steps\n        super()._on_training_end()\n        summary = json.loads((self.output/\'training_summary.json\').read_text())\n        summary.update(optimizer_steps_requested=self.args.optimizer_steps,\n            exact_update_budget_met=self.actual_gradient_steps == self.args.optimizer_steps,\n            target_kl=self.args.target_kl, completed_rollouts=self.model.completed_rollouts,\n            optimizer_updates_counted_by=\'Adam post-step hook, checked against explicit counter\')\n        write_json(self.output/\'training_summary.json\', summary)\n\n\nclass EqualBudgetPPO(BalancedMaskablePPO):\n    """A hard budget limits genuine Adam steps, including the last partial epoch."""\n\n    def __init__(self, *args, **kwargs):\n        self.actual_optimizer_steps = 0\n        self.rollout_optimizer_budget = 0\n        self.completed_rollouts = 0\n        self.budget_callback = None\n        self.budget_output = None\n        super().__init__(*args, **kwargs)\n\n    def _excluded_save_params(self):\n        return super()._excluded_save_params() + [\'budget_callback\']\n\n    def train(self):\n        self.policy.set_training_mode(True)\n        self._update_learning_rate(self.policy.optimizer)\n        clip_range = self.clip_range(self._current_progress_remaining)\n        clip_vf = (self.clip_range_vf(self._current_progress_remaining)\n                   if self.clip_range_vf is not None else None)\n        requested = int(self.rollout_optimizer_budget)\n        assert requested > 0\n        size = self.rollout_buffer.buffer_size * self.n_envs\n        batches_per_epoch = math.ceil(size/self.batch_size)\n        max_epochs = math.ceil(requested/batches_per_epoch)\n        stats = {name: [] for name in [\'approx_kl\', \'unweighted_approx_kl\',\n            \'policy_loss\', \'value_loss\', \'entropy_loss\', \'clip_fraction\', \'gradient_norm\']}\n        before = self.actual_optimizer_steps\n        completed_epochs = 0\n        stopped_by_kl = False\n        for epoch in range(max_epochs):\n            epoch_batches = 0\n            for data in self.rollout_buffer.get(self.batch_size):\n                if self.actual_optimizer_steps-before >= requested:\n                    break\n                actions = data.actions.long().flatten() if isinstance(self.action_space, spaces.Discrete) else data.actions\n                values, log_prob, entropy = self.policy.evaluate_actions(\n                    data.observations, actions, action_masks=data.action_masks)\n                values = values.flatten()\n                advantages = data.advantages\n                if self.normalize_advantage:\n                    advantages = (advantages-advantages.mean())/(advantages.std()+1e-8)\n                log_ratio = log_prob-data.old_log_prob\n                ratio = th.exp(log_ratio)\n                weights = data.actor_weights\n                policy_loss = clipped_actor_loss(advantages, ratio, weights, clip_range)\n                value_prediction = values if clip_vf is None else data.old_values + th.clamp(\n                    values-data.old_values, -clip_vf, clip_vf)\n                # Same unweighted critic as all previous actor-only ablations.\n                value_loss = F.mse_loss(data.returns, value_prediction)\n                entropy_loss = -weighted_mean(-log_prob if entropy is None else entropy, weights)\n                loss = policy_loss+self.ent_coef*entropy_loss+self.vf_coef*value_loss\n                with th.no_grad():\n                    kl_terms = (th.exp(log_ratio)-1)-log_ratio\n                    kl = weighted_mean(kl_terms, weights).item()\n                    unweighted_kl = kl_terms.mean().item()\n                    fraction = weighted_mean(((ratio-1).abs()>clip_range).float(), weights).item()\n                if not (th.isfinite(loss) and np.isfinite(kl)):\n                    raise FloatingPointError(\'Nonfinite PPO loss/KL; do not count a failed update\')\n                if self.target_kl is not None and kl > 1.5*self.target_kl:\n                    stopped_by_kl = True\n                    break\n                self.policy.optimizer.zero_grad()\n                loss.backward()\n                norm = th.nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)\n                if not th.isfinite(norm):\n                    raise FloatingPointError(\'Nonfinite gradient; do not count a failed update\')\n                self.policy.optimizer.step()\n                self.actual_optimizer_steps += 1\n                epoch_batches += 1\n                for name, value in dict(approx_kl=kl, unweighted_approx_kl=unweighted_kl,\n                    policy_loss=policy_loss.item(), value_loss=value_loss.item(),\n                    entropy_loss=entropy_loss.item(), clip_fraction=fraction,\n                    gradient_norm=norm.item()).items():\n                    stats[name].append(value)\n                if self.budget_callback is not None:\n                    self.budget_callback.after_optimizer_step()\n            # Epochs are separate from actual optimizer steps, including partial epochs.\n            self._n_updates += 1\n            if epoch_batches == batches_per_epoch:\n                completed_epochs += 1\n            if stopped_by_kl or self.actual_optimizer_steps-before >= requested:\n                break\n        self.completed_rollouts += 1\n        performed = self.actual_optimizer_steps-before\n        if self.target_kl is None:\n            assert performed == requested\n        row = dict(rollout=self.completed_rollouts, env_steps=int(self.num_timesteps),\n            rollout_transitions=size, requested_optimizer_steps=requested,\n            actual_optimizer_steps=performed, cumulative_optimizer_steps=self.actual_optimizer_steps,\n            full_epochs=completed_epochs, partial_epoch_batches=performed % batches_per_epoch,\n            target_kl=self.target_kl, stopped_by_kl=stopped_by_kl,\n            approx_kl_first=stats[\'approx_kl\'][0] if performed else None,\n            approx_kl_last=stats[\'approx_kl\'][-1] if performed else None,\n            approx_kl_max=max(stats[\'approx_kl\']) if performed else None,\n            fraction_updates_kl_above_previous_limit=float(np.mean(np.array(stats[\'approx_kl\'])>.045)) if performed else None,\n            **{name+\'_mean\':float(np.mean(values)) if values else None for name,values in stats.items()})\n        if self.budget_output is not None:\n            with (Path(self.budget_output)/\'optimizer_diagnostics.jsonl\').open(\'a\') as f:\n                f.write(json.dumps(row, allow_nan=False)+\'\\n\')\n        for name, values in stats.items():\n            if values: self.logger.record(\'train/\'+name, np.mean(values))\n        self.logger.record(\'train/actual_optimizer_steps\', self.actual_optimizer_steps)\n        self.logger.record(\'train/explained_variance\', explained_variance(\n            self.rollout_buffer.values.flatten(), self.rollout_buffer.returns.flatten()))\n\n    def learn_exact(self, total_interactions, total_optimizer_steps, callback):\n        if total_interactions % self.n_envs:\n            raise ValueError(\'Exact interaction budget must be divisible by n_envs\')\n        if total_optimizer_steps != total_interactions:\n            raise ValueError(\'This controlled runner is specifically one optimizer step per interaction\')\n        total_interactions, callback = self._setup_learn(total_interactions, callback,\n            reset_num_timesteps=True, tb_log_name=\'EqualBudgetPPO\', progress_bar=False)\n        self.budget_callback = callback\n        self.budget_output = str(callback.output)\n        callback.on_training_start(locals(), globals())\n        original_n_steps = self.n_steps\n        try:\n            while self.num_timesteps < total_interactions:\n                remaining = total_interactions-self.num_timesteps\n                steps = min(original_n_steps, remaining//self.n_envs)\n                if self.rollout_buffer.buffer_size != steps:\n                    self.rollout_buffer = BalancedRolloutBuffer(steps, self.observation_space,\n                        self.action_space, device=self.device, gae_lambda=self.gae_lambda,\n                        gamma=self.gamma, n_envs=self.n_envs, **self.rollout_buffer_kwargs)\n                before = self.num_timesteps\n                if not self.collect_rollouts(self.env, callback, self.rollout_buffer,\n                                             n_rollout_steps=steps, use_masking=True):\n                    raise RuntimeError(\'Collection stopped before the requested budget\')\n                collected = self.num_timesteps-before\n                assert collected == steps*self.n_envs\n                self.rollout_optimizer_budget = collected\n                self._update_current_progress_remaining(self.num_timesteps, total_interactions)\n                self.train()\n                self.logger.dump(step=self.num_timesteps)\n            assert self.num_timesteps == total_interactions\n            if self.target_kl is None:\n                assert self.actual_optimizer_steps == total_optimizer_steps\n            callback.on_training_end()\n        finally:\n            self.budget_callback = None\n        return self\n',encoding='utf-8')
print('Exact source ready: equal_budget_ppo.py')

Exact source ready: equal_budget_ppo.py


In [6]:
(PROJECT/'run_equal_budget_experiments.py').write_text('"""Five predeclared Wafer experiments, each trained from scratch on TRAIN only.\n\nThe common environment, feature encoder and PPO losses are copied from the\ncompleted alpha=.5 reference. EXPERIMENT fixes the one ablation in this folder.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport platform\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom gymnasium import spaces\nfrom torch import nn\nfrom sklearn.model_selection import train_test_split\nfrom sb3_contrib import MaskablePPO\nfrom stable_baselines3.common.torch_layers import FlattenExtractor\nfrom stable_baselines3.common.vec_env import DummyVecEnv\n\nfrom balanced_ppo import BalancedMaskablePPO, BalancedRolloutBuffer, inverse_frequency_weights, weighted_mean\nfrom equal_budget_ppo import EqualBudgetPPO, UpdateEvaluateCallback\nfrom ppo_wafer import PrefixCNN, PrefixWaferEnv, EvaluateCallback, evaluate, load_split, write_json, HASHES\n\nEXPERIMENT = {\'name\': \'ppo_alpha05_start1\', \'kind\': \'start1\', \'alpha\': 0.5, \'description\': \'Current 100k/100k PPO, alpha=.5, retaining the complete observed-prefix CNN; change only training episode initialization to always t=1.\'}\nROOT = Path(__file__).resolve().parent\n\n\nclass CurrentOnlyWaferEnv(PrefixWaferEnv):\n    """Only x_t is observed: no history, position, encoder, RNN or substitute."""\n    def __init__(self, *args, **kwargs):\n        super().__init__(*args, **kwargs)\n        self.observation_space = spaces.Box(-np.inf, np.inf, (1,), np.float32)\n\n    def observation(self):\n        return self.X[self.index, self.t-1:self.t].copy()\n\n\nclass LegacyAuditMixin:\n    """Instrument the unchanged legacy train() without modifying its updates."""\n    def train(self):\n        before = self.actual_optimizer_steps\n        super().train()\n        self.completed_rollouts += 1\n        row = dict(rollout=self.completed_rollouts, env_steps=int(self.num_timesteps),\n            rollout_transitions=int(self.rollout_buffer.buffer_size*self.n_envs),\n            actual_optimizer_steps=self.actual_optimizer_steps-before,\n            cumulative_optimizer_steps=self.actual_optimizer_steps,\n            nominal_epochs=self.n_epochs, target_kl=self.target_kl,\n            maximum_optimizer_steps=self.n_epochs*math.ceil(self.rollout_buffer.buffer_size*self.n_envs/self.batch_size))\n        row[\'stopped_by_kl\'] = row[\'actual_optimizer_steps\'] < row[\'maximum_optimizer_steps\']\n        with (Path(self.audit_output)/\'optimizer_diagnostics.jsonl\').open(\'a\') as stream:\n            stream.write(json.dumps(row, allow_nan=False)+\'\\n\')\n\n\nclass LegacyMaskablePPO(LegacyAuditMixin, MaskablePPO):\n    pass\n\n\nclass LegacyBalancedPPO(LegacyAuditMixin, BalancedMaskablePPO):\n    pass\n\n\nclass LegacyCallback(EvaluateCallback):\n    def _on_training_start(self):\n        super()._on_training_start()\n        def count_model(optimizer, args, kwargs):\n            self.model.actual_optimizer_steps += 1\n        self.model_counter_hook = self.model.policy.optimizer.register_step_post_hook(count_model)\n        self.saved_100k = False\n\n    def _on_step(self):\n        result = super()._on_step()\n        if self.num_timesteps == 100000 and not self.saved_100k:\n            # Read-only snapshot during collection: it does not shorten the old\n            # rollout or introduce an extra update into the old training recipe.\n            assert self.actual_gradient_steps == self.model.actual_optimizer_steps\n            self.model.save(self.output/\'budget100k_model\')\n            validation = evaluate(self.model, self.X, self.y, self.args.wait_lambda, self.args.wait_power)\n            write_json(self.output/\'budget100k_metadata.json\', dict(\n                env_steps=100000, actual_gradient_steps=self.actual_gradient_steps,\n                latest_completed_training_rollout_env_steps=98304,\n                snapshot=\'during collection, before the next legacy PPO update\',\n                checkpoint_selection=\'fixed 100000 interactions, not validation selected\',\n                validation=validation))\n            self.saved_100k = True\n            print(\'[fixed 100k snapshot]\', json.dumps(validation), flush=True)\n        return result\n\n    def _on_training_end(self):\n        assert self.saved_100k\n        assert self.actual_gradient_steps == self.model.actual_optimizer_steps\n        super()._on_training_end()\n        self.model_counter_hook.remove()\n\n\nclass FixedPrefixClassifier(nn.Module):\n    """Same prefix CNN and actor hidden layer; two class logits, no RL/critic."""\n    def __init__(self):\n        super().__init__()\n        observation_space = spaces.Box(-np.inf, np.inf, (152,), np.float32)\n        self.features_extractor = PrefixCNN(observation_space, features_dim=128)\n        self.class_hidden = nn.Sequential(nn.Linear(128, 64), nn.Tanh())\n        self.class_output = nn.Linear(64, 2)\n        # Same orthogonal initialization gains as the PPO actor.\n        for module, gain in [(self.features_extractor, np.sqrt(2)),\n                             (self.class_hidden, np.sqrt(2)), (self.class_output, .01)]:\n            def initialize(layer, gain=gain):\n                if isinstance(layer, (nn.Linear, nn.Conv1d)):\n                    nn.init.orthogonal_(layer.weight, gain=gain)\n                    if layer.bias is not None: nn.init.zeros_(layer.bias)\n            module.apply(initialize)\n\n    def forward(self, observation):\n        return self.class_output(self.class_hidden(self.features_extractor(observation)))\n\n\ndef fixed_prefix_observations(X):\n    observation = np.zeros_like(X, dtype=np.float32)\n    observation[:, :2] = X[:, :2]\n    return observation\n\n\ndef versions():\n    return dict(python=platform.python_version(), torch=torch.__version__,\n        sb3=__import__(\'stable_baselines3\').__version__,\n        sb3_contrib=__import__(\'sb3_contrib\').__version__,\n        gymnasium=__import__(\'gymnasium\').__version__)\n\n\ndef preflight(X, y, train_ids, val_ids):\n    reference = json.loads((ROOT/\'PROVENANCE.json\').read_text())[\'reference_split\']\n    assert train_ids.tolist() == reference[\'train_indices\']\n    assert val_ids.tolist() == reference[\'monitor_indices\']\n    assert np.bincount(y[train_ids], minlength=2).tolist() == [78, 722]\n    assert np.bincount(y[val_ids], minlength=2).tolist() == [19, 181]\n    env = CurrentOnlyWaferEnv(X, y, balanced=False)\n    obs, _ = env.reset(options=dict(index=0, start_at=10))\n    changed = X.copy(); changed[0, :9] = 123456.; changed[0, 10:] = -987654.\n    other = CurrentOnlyWaferEnv(changed, y, balanced=False)\n    altered, _ = other.reset(options=dict(index=0, start_at=10))\n    assert obs.shape == (1,) and np.array_equal(obs, altered) and obs[0] == X[0, 9]\n    next_obs, _, done, _, _ = env.step(0)\n    assert not done and next_obs[0] == X[0, 10]\n    env.reset(options=dict(index=0, start_at=152))\n    assert env.action_masks().tolist() == [False, True, True]\n    env = PrefixWaferEnv(X, y, balanced=False, random_start_fraction=0.)\n    assert all(env.reset()[0].shape == (152,) and env.t == 1 for _ in range(100))\n    assert np.count_nonzero(fixed_prefix_observations(X)[:, 2:]) == 0\n    print(\'[preflight] exact split, current-only absence of past/future, WAIT mask, start-at-one and two-point input passed\', flush=True)\n\n\ndef train_static(seed, directory, X, y, train_ids, val_ids, args):\n    model = FixedPrefixClassifier()\n    assert sum(p.numel() for p in model.parameters()) == 87042\n    weights = torch.as_tensor(inverse_frequency_weights(y[train_ids], .5), dtype=torch.float32)\n    features = torch.from_numpy(fixed_prefix_observations(X))\n    labels = torch.as_tensor(y, dtype=torch.long)\n    training_features, training_labels = features[train_ids], labels[train_ids]\n    validation_features, validation_labels = features[val_ids], labels[val_ids]\n    optimizer = torch.optim.Adam(model.parameters(), lr=.0003, eps=1e-5)\n    started = time.monotonic()\n    best_loss, best_epoch, stale, updates = float(\'inf\'), 0, 0, 0\n    selected_path = directory/\'selected_model.pt\'\n    for epoch in range(1, 257):\n        model.train()\n        permutation = torch.randperm(len(train_ids))\n        training_loss = 0.\n        for batch in permutation.split(256):\n            logits = model(training_features[batch])\n            per_item = nn.functional.cross_entropy(logits, training_labels[batch], reduction=\'none\')\n            loss = weighted_mean(per_item, weights[training_labels[batch]])\n            if not torch.isfinite(loss): raise FloatingPointError(\'Nonfinite supervised loss\')\n            optimizer.zero_grad(); loss.backward()\n            norm = nn.utils.clip_grad_norm_(model.parameters(), .5)\n            if not torch.isfinite(norm): raise FloatingPointError(\'Nonfinite supervised gradient\')\n            optimizer.step(); updates += 1\n            training_loss += loss.item()*len(batch)/len(train_ids)\n        model.eval()\n        with torch.no_grad():\n            logits = model(validation_features)\n            losses = nn.functional.cross_entropy(logits, validation_labels, reduction=\'none\')\n            validation_loss = weighted_mean(losses, weights[validation_labels]).item()\n            predictions = logits.argmax(1).numpy()\n        from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score\n        improved = validation_loss < best_loss-1e-8\n        if improved:\n            best_loss, best_epoch, stale = validation_loss, epoch, 0\n            torch.save(dict(state_dict=model.state_dict(), seed=seed, selected_epoch=epoch,\n                optimizer_steps_at_selection=updates, selection=\'minimum weighted validation cross-entropy\',\n                validation_loss=validation_loss, input_prefix_length=2, model_parameters=87042), selected_path)\n        else: stale += 1\n        row = dict(epoch=epoch, actual_gradient_steps=updates, training_loss=training_loss,\n            validation_loss=validation_loss, accuracy=float(accuracy_score(y[val_ids], predictions)),\n            balanced_accuracy=float(balanced_accuracy_score(y[val_ids], predictions)),\n            macro_f1=float(f1_score(y[val_ids], predictions, average=\'macro\', zero_division=0)),\n            avg_t=2., selected_epoch=best_epoch, seconds=time.monotonic()-started)\n        with (directory/\'monitor.jsonl\').open(\'a\') as stream: stream.write(json.dumps(row)+\'\\n\')\n        if epoch % 10 == 0 or epoch == 1: print(\'[supervised validation]\', json.dumps(row), flush=True)\n        if stale >= 30: break\n    selected = torch.load(selected_path, weights_only=False, map_location=\'cpu\')\n    assert selected[\'selected_epoch\'] == best_epoch\n    write_json(directory/\'training_summary.json\', dict(completed=True, seed=seed,\n        algorithm=\'supervised fixed-prefix classifier; no PPO, WAIT, value function or environment interactions\',\n        env_steps=None, actual_gradient_steps=updates, completed_epochs=epoch,\n        selected_epoch=best_epoch, optimizer_steps_at_selection=selected[\'optimizer_steps_at_selection\'],\n        checkpoint_selection=\'minimum weighted validation cross-entropy; patience 30, maximum 256 epochs\',\n        model_parameters=87042, wall_seconds=time.monotonic()-started))\n    print(\'[completed supervised]\', directory, flush=True)\n\n\ndef train_one(seed, output=None, threads=2):\n    torch.set_num_threads(threads)\n    torch.use_deterministic_algorithms(True)\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    X, y = load_split(ROOT/\'data\', \'TRAIN\')\n    train_ids, val_ids = train_test_split(np.arange(len(y)), test_size=.2, stratify=y, random_state=0)\n    preflight(X, y, train_ids, val_ids)\n    spec = EXPERIMENT\n    directory = Path(output) if output else ROOT/\'models\'/f\'seed{seed}\'\n    directory.mkdir(parents=True, exist_ok=True)\n    if (directory/\'config.json\').exists(): raise FileExistsError(\'Fresh training requires an empty output directory\')\n    legacy, static = spec[\'kind\'] == \'legacy\', spec[\'kind\'] == \'static\'\n    alpha = float(spec.get(\'alpha\', .5))\n    weights = inverse_frequency_weights(y[train_ids], alpha)\n    steps = 1048576 if legacy else (None if static else 100000)\n    args = argparse.Namespace(experiment=spec[\'name\'], mode=\'train\', protocol=\'holdout\', seed=seed,\n        split_seed=0, train_order=\'split\', steps=steps,\n        optimizer_steps=None if legacy or static else 100000, natural_sampling=True,\n        n_envs=None if static else 32, n_steps=None if static else 64, batch_size=256,\n        epochs=256 if not legacy else 5, gae_lambda=.95, lr=.0003,\n        ent_coef=.02, initial_wait_bias=0., init_checkpoint=None, exploration_mix=0.,\n        wait_lambda=.0001, wait_power=1/3, correct_reward=1., incorrect_reward=-1.,\n        actor_balance_alpha=None if static else alpha, supervised_class_balance_alpha=.5 if static else None,\n        balance_value_loss=False, reward_balance_alpha=0., reward_class_weights=[1., 1.],\n        random_start_fraction=0. if spec[\'kind\']==\'start1\' else .5, threads=threads,\n        sampler=\'standard\', max_validation_mean_t=0., save_reference_at=0,\n        eval_every=32768 if legacy else 5000,\n        eval_every_unit=\'environment_interactions\' if legacy else \'actual_optimizer_steps\',\n        target_kl=.03 if legacy else None,\n        data_dir=str(ROOT/\'data\'), output=str(directory), exact_budget=not legacy and not static,\n        observation_mode=\'current_scalar_only\' if spec[\'kind\']==\'current_only\' else \'right_zero_padded_observed_prefix\',\n        input_prefix_length=2 if static else None, algorithm=spec[\'description\'])\n    if static:\n        args.static_max_epochs=256; args.static_early_stopping_patience=30\n        args.static_checkpoint_selection=\'minimum weighted validation cross-entropy\'\n        args.random_start_fraction=None; args.ent_coef=None; args.gae_lambda=None\n    write_json(directory/\'config.json\', vars(args))\n    write_json(directory/\'split_manifest.json\', dict(dataset=\'original UCR2015 Wafer\',\n        train_hash=HASHES[\'TRAIN\'], train_indices=train_ids.tolist(), monitor_indices=val_ids.tolist(),\n        monitor=\'independent_validation\', test_used_for_selection=False, observation=args.observation_mode,\n        actor_class_weights=None if static else weights.tolist(),\n        supervised_class_weights=weights.tolist() if static else None,\n        actor_balance_alpha=args.actor_balance_alpha, sampler=\'standard\',\n        reward=dict(correct=1., incorrect=-1., wait_lambda=.0001, wait_power=1/3, class_weights=[1., 1.]),\n        classification_pretraining=False, auxiliary_classification_loss=False,\n        initialization=\'fresh random weights; no historical checkpoints reused\', versions=versions()))\n    if static:\n        train_static(seed, directory, X, y, train_ids, val_ids, args)\n        return\n    env_class = CurrentOnlyWaferEnv if spec[\'kind\']==\'current_only\' else PrefixWaferEnv\n    def make_env(i):\n        return lambda: env_class(X[train_ids], y[train_ids], seed=seed+i, wait_lambda=.0001,\n            wait_power=1/3, balanced=False, random_start_fraction=args.random_start_fraction,\n            sampler=\'standard\')\n    vec = DummyVecEnv([make_env(i) for i in range(32)])\n    policy_kwargs = dict(net_arch=dict(pi=[64], vf=[64]), activation_fn=nn.Tanh)\n    if spec[\'kind\']==\'current_only\':\n        policy_kwargs[\'features_extractor_class\'] = FlattenExtractor\n    else:\n        policy_kwargs.update(features_extractor_class=PrefixCNN, features_extractor_kwargs=dict(features_dim=128))\n    balanced = not legacy or alpha != 0.\n    model_class = (LegacyBalancedPPO if alpha else LegacyMaskablePPO) if legacy else EqualBudgetPPO\n    extra = dict(rollout_buffer_class=BalancedRolloutBuffer,\n        rollout_buffer_kwargs=dict(class_weights=weights)) if balanced else {}\n    model = model_class(\'MlpPolicy\', vec, learning_rate=.0003, n_steps=64, batch_size=256,\n        n_epochs=args.epochs, gamma=1., gae_lambda=.95, clip_range=.2, ent_coef=.02,\n        vf_coef=.5, max_grad_norm=.5, target_kl=args.target_kl, seed=seed,\n        device=\'cpu\', verbose=0, policy_kwargs=policy_kwargs, **extra)\n    parameters = sum(p.numel() for p in model.policy.parameters())\n    assert parameters == (516 if spec[\'kind\']==\'current_only\' else 95428)\n    assert sum(p.numel() for p in model.policy.features_extractor.parameters()) == (0 if spec[\'kind\']==\'current_only\' else 78656)\n    if legacy:\n        model.actual_optimizer_steps=0; model.completed_rollouts=0; model.audit_output=str(directory)\n    callback_class = LegacyCallback if legacy else UpdateEvaluateCallback\n    callback = callback_class(X[val_ids], y[val_ids], directory, args)\n    print(\'[start]\', spec[\'name\'], \'seed\', seed, \'interactions\', steps, \'parameters\', parameters,\n          \'alpha\', alpha, \'epochs\', args.epochs, \'target_kl\', args.target_kl, flush=True)\n    started = time.monotonic()\n    try:\n        if legacy: model.learn(total_timesteps=steps, callback=callback)\n        else: model.learn_exact(100000, 100000, callback)\n    finally:\n        vec.close()\n    summary = json.loads((directory/\'training_summary.json\').read_text())\n    adam_steps = {int(s[\'step\']) for s in model.policy.optimizer.state.values() if \'step\' in s}\n    assert adam_steps == {summary[\'actual_gradient_steps\']}\n    assert summary[\'env_steps\'] == steps\n    summary.update(seed=seed, wall_seconds=time.monotonic()-started, target_kl=args.target_kl,\n        completed_rollouts=model.completed_rollouts, epochs_per_full_rollout=args.epochs,\n        actual_adam_state_steps=list(adam_steps), fresh_random_initialization=True,\n        fixed_final_checkpoint=True, observation_mode=args.observation_mode)\n    write_json(directory/\'training_summary.json\', summary)\n    for name in [\'best_model.zip\', \'best_monitor.json\', \'early_best_model.zip\', \'early_best_monitor.json\']:\n        path=directory/name\n        if path.exists(): path.unlink()\n    print(\'[completed]\', spec[\'name\'], seed, json.dumps(summary), flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--seed\', type=int, required=True)\n    parser.add_argument(\'--output\')\n    parser.add_argument(\'--threads\', type=int, default=2)\n    a = parser.parse_args()\n    train_one(a.seed, a.output, a.threads)\n',encoding='utf-8')
print('Exact source ready: run_equal_budget_experiments.py')

Exact source ready: run_equal_budget_experiments.py


In [7]:
(PROJECT/'evaluate_final.py').write_text('"""Lock trained checkpoints, then evaluate validation and official TEST.\n\nAll choices are predeclared in EXPERIMENT and training configs. TEST never\nchooses a checkpoint or alters training. Prediction files retain both splits.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, matthews_corrcoef, cohen_kappa_score\n\nfrom equal_budget_ppo import EqualBudgetPPO\nfrom ppo_wafer import evaluate, load_split, write_json\nfrom run_equal_budget_experiments import EXPERIMENT, LegacyMaskablePPO, LegacyBalancedPPO, FixedPrefixClassifier, fixed_prefix_observations\n\nROOT=Path(__file__).resolve().parent\nsha=lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()\n\n\ndef prediction_metrics(labels, predictions, times):\n    cm=confusion_matrix(labels, predictions, labels=[0, 1])\n    costs=np.r_[0., np.cumsum(.0001*np.arange(1, 152)**(1/3))]\n    return dict(n=len(labels), evaluation=\'greedy\', accuracy=float(accuracy_score(labels,predictions)),\n        balanced_accuracy=float(balanced_accuracy_score(labels,predictions)),\n        macro_f1=float(f1_score(labels,predictions,average=\'macro\',zero_division=0)),\n        weighted_f1=float(f1_score(labels,predictions,average=\'weighted\',zero_division=0)),\n        minority_f1=float(f1_score(labels,predictions,pos_label=0,zero_division=0)),\n        minority_precision=float(precision_score(labels,predictions,pos_label=0,zero_division=0)),\n        minority_recall=float(recall_score(labels,predictions,pos_label=0,zero_division=0)),\n        majority_recall=float(recall_score(labels,predictions,pos_label=1,zero_division=0)),\n        precision_by_class=precision_score(labels,predictions,labels=[0,1],average=None,zero_division=0).tolist(),\n        f1_by_class=f1_score(labels,predictions,labels=[0,1],average=None,zero_division=0).tolist(),\n        recall_by_class=recall_score(labels,predictions,labels=[0,1],average=None,zero_division=0).tolist(),\n        mcc=float(matthews_corrcoef(labels,predictions)), cohen_kappa=float(cohen_kappa_score(labels,predictions)),\n        confusion_matrix=cm.tolist(), avg_t=float(times.mean()), earliness=float(times.mean()/152),\n        first_stop_fraction=float(np.mean(times==1)),\n        mean_return=float(np.mean(np.where(labels==predictions,1.,-1.)-costs[times-1])),\n        avg_t_by_class=[float(times[labels==c].mean()) for c in (0,1)],\n        decision_time_median=float(np.median(times)), decision_time_p90=float(np.quantile(times,.9)),\n        decision_time_p95=float(np.quantile(times,.95)), decision_time_max=int(times.max()),\n        predicted_class_counts=np.bincount(predictions,minlength=2).tolist(),\n        single_class_predictions=bool(len(np.unique(predictions))==1))\n\n\ndef aggregate(rows):\n    keys=[\'accuracy\',\'balanced_accuracy\',\'macro_f1\',\'weighted_f1\',\'minority_f1\',\n        \'minority_precision\',\'minority_recall\',\'majority_recall\',\'mcc\',\'cohen_kappa\',\n        \'avg_t\',\'earliness\',\'first_stop_fraction\',\'mean_return\',\'decision_time_median\',\n        \'decision_time_p90\',\'decision_time_p95\',\'decision_time_max\']\n    out={k:dict(mean=float(np.mean([r[k] for r in rows])),\n                sample_sd=float(np.std([r[k] for r in rows],ddof=1))) for k in keys}\n    out[\'avg_t_minority\']=dict(mean=float(np.mean([r[\'avg_t_by_class\'][0] for r in rows])),\n        sample_sd=float(np.std([r[\'avg_t_by_class\'][0] for r in rows],ddof=1)))\n    out[\'avg_t_majority\']=dict(mean=float(np.mean([r[\'avg_t_by_class\'][1] for r in rows])),\n        sample_sd=float(np.std([r[\'avg_t_by_class\'][1] for r in rows],ddof=1)))\n    return out\n\n\ndef main(models_root, data_dir, threads=2):\n    torch.set_num_threads(threads)\n    spec=EXPERIMENT\n    static, legacy=spec[\'kind\']==\'static\',spec[\'kind\']==\'legacy\'\n    locks=[]\n    for seed in [0,1,2]:\n        directory=models_root/f\'seed{seed}\'\n        config=json.loads((directory/\'config.json\').read_text())\n        summary=json.loads((directory/\'training_summary.json\').read_text())\n        manifest=json.loads((directory/\'split_manifest.json\').read_text())\n        reference=json.loads((ROOT/\'PROVENANCE.json\').read_text())[\'reference_split\']\n        assert manifest[\'train_indices\']==reference[\'train_indices\'] and manifest[\'monitor_indices\']==reference[\'monitor_indices\']\n        assert summary[\'completed\'] and summary[\'fresh_random_initialization\'] if not static else summary[\'completed\']\n        assert config[\'seed\']==seed and config[\'experiment\']==spec[\'name\']\n        assert config[\'wait_lambda\']==.0001 and config[\'reward_class_weights\']==[1.,1.]\n        if not static:\n            assert summary[\'env_steps\']==(1048576 if legacy else 100000)\n            assert summary[\'actual_gradient_steps\']==summary[\'actual_adam_state_steps\'][0]\n            if not legacy: assert summary[\'actual_gradient_steps\']==100000\n        checkpoints=[(\'selected\' if static else \'final\', \'selected_model.pt\' if static else \'last_model.zip\')]\n        if legacy: checkpoints.append((\'budget100k\',\'budget100k_model.zip\'))\n        for stage, filename in checkpoints:\n            path=directory/filename\n            assert path.is_file()\n            locks.append(dict(seed=seed,stage=stage,checkpoint=filename,checkpoint_sha256=sha(path),\n                checkpoint_selection=\'minimum validation cross-entropy\' if static else \'fixed predeclared interaction budget\',\n                TEST_used_for_selection=False))\n        write_json(directory/\'selection_lock.json\',[r for r in locks if r[\'seed\']==seed])\n    # The official TEST is first opened after every required checkpoint in this\n    # experiment has been frozen by hash. Validation never changes these locks.\n    X_train,y_train=load_split(data_dir,\'TRAIN\')\n    X_test,y_test=load_split(data_dir,\'TEST\')\n    runs=[]; csv_rows=[]; checks=[]\n    for item in locks:\n        seed,stage=item[\'seed\'],item[\'stage\']\n        directory=models_root/f\'seed{seed}\'\n        config=json.loads((directory/\'config.json\').read_text())\n        summary=json.loads((directory/\'training_summary.json\').read_text())\n        manifest=json.loads((directory/\'split_manifest.json\').read_text())\n        path=directory/item[\'checkpoint\']\n        assert sha(path)==item[\'checkpoint_sha256\']\n        if static:\n            selected=torch.load(path,map_location=\'cpu\',weights_only=False)\n            model=FixedPrefixClassifier();model.load_state_dict(selected[\'state_dict\']);model.eval()\n            optimizer_steps=selected[\'optimizer_steps_at_selection\'];env_steps=None\n        else:\n            cls=(LegacyBalancedPPO if spec[\'alpha\'] else LegacyMaskablePPO) if legacy else EqualBudgetPPO\n            model=cls.load(path,device=\'cpu\')\n            env_steps=int(model.num_timesteps);optimizer_steps=int(model.actual_optimizer_steps)\n            assert env_steps==(100000 if stage==\'budget100k\' or not legacy else 1048576)\n            assert model.n_steps==64 and model.batch_size==256 and model.n_envs==32\n            assert model.n_epochs==(5 if legacy else 256) and model.target_kl==(.03 if legacy else None)\n            assert model.gamma==1. and model.gae_lambda==.95 and model.ent_coef==.02 and model.vf_coef==.5\n            assert model.normalize_advantage and model.clip_range_vf is None\n            assert model.clip_range(1)==.2 and model.max_grad_norm==.5\n            assert model.policy.net_arch==dict(pi=[64],vf=[64])\n            adam_steps={int(s[\'step\']) for s in model.policy.optimizer.state.values() if \'step\' in s}\n            assert adam_steps=={optimizer_steps}\n            if spec[\'kind\']==\'current_only\':\n                assert model.observation_space.shape==(1,)\n                assert sum(p.numel() for p in model.policy.features_extractor.parameters())==0\n            else: assert model.observation_space.shape==(152,)\n        parameters=sum(p.numel() for p in (model.parameters() if static else model.policy.parameters()))\n        assert parameters==(87042 if static else (516 if spec[\'kind\']==\'current_only\' else 95428))\n        assert all(torch.isfinite(p).all() for p in (model.parameters() if static else model.policy.parameters()))\n        record=dict(**item,environment_steps=env_steps,optimizer_steps=optimizer_steps,model_parameters=parameters)\n        for split,X,y in [(\'validation\',X_train[np.asarray(manifest[\'monitor_indices\'])],y_train[np.asarray(manifest[\'monitor_indices\'])]),(\'test\',X_test,y_test)]:\n            if static:\n                with torch.no_grad():\n                    probabilities=model(torch.from_numpy(fixed_prefix_observations(X))).softmax(-1).numpy()\n                predictions=dict(labels=y.copy(),predictions=probabilities.argmax(1),\n                    decision_times=np.full(len(y),2,dtype=np.int64),class_probabilities=probabilities)\n                original=None\n            else:\n                original,predictions=evaluate(model,X,y,.0001,1/3,stochastic=False,seed=seed+10000,return_predictions=True)\n            labels=predictions[\'labels\'].astype(int)\n            predicted=predictions[\'predictions\'].astype(int)\n            times=predictions[\'decision_times\'].astype(int)\n            assert np.array_equal(labels,y) and np.all((times>=1)&(times<=152))\n            metrics=prediction_metrics(labels,predicted,times)\n            if original:\n                for key in [\'accuracy\',\'balanced_accuracy\',\'macro_f1\',\'confusion_matrix\',\'recall_by_class\',\'avg_t\',\'earliness\',\'first_stop_fraction\',\'mean_return\',\'avg_t_by_class\']:\n                    assert np.allclose(metrics[key],original[key],rtol=1e-10,atol=1e-10),(seed,stage,split,key)\n                metrics[\'action_counts\']=original[\'action_counts\']\n                metrics[\'first_probs_by_class\']=original[\'first_probs_by_class\']\n            prefix=\'\' if stage in [\'final\',\'selected\'] else stage+\'_\'\n            np.savez_compressed(directory/f\'{prefix}{split}_predictions_greedy.npz\',**predictions)\n            record[split]=metrics\n            write_json(directory/f\'{prefix}final_{split}_results.json\',dict(seed=seed,stage=stage,\n                checkpoint=item[\'checkpoint\'],checkpoint_sha256=item[\'checkpoint_sha256\'],\n                environment_steps=env_steps,optimizer_steps=optimizer_steps,greedy=metrics))\n            row=dict(experiment=spec[\'name\'],seed=seed,stage=stage,split=split,\n                checkpoint=item[\'checkpoint\'],environment_steps=env_steps,optimizer_steps=optimizer_steps,\n                model_parameters=parameters,accuracy_pct=100*metrics[\'accuracy\'],\n                balanced_accuracy_pct=100*metrics[\'balanced_accuracy\'],macro_f1_pct=100*metrics[\'macro_f1\'],\n                weighted_f1_pct=100*metrics[\'weighted_f1\'],minority_f1_pct=100*metrics[\'minority_f1\'],\n                minority_precision_pct=100*metrics[\'minority_precision\'],minority_recall_pct=100*metrics[\'minority_recall\'],\n                majority_recall_pct=100*metrics[\'majority_recall\'],mcc=metrics[\'mcc\'],cohen_kappa=metrics[\'cohen_kappa\'],\n                mean_observations=metrics[\'avg_t\'],earliness_pct=100*metrics[\'earliness\'],\n                first_stop_pct=100*metrics[\'first_stop_fraction\'],minority_mean_observations=metrics[\'avg_t_by_class\'][0],\n                majority_mean_observations=metrics[\'avg_t_by_class\'][1],mean_return=metrics[\'mean_return\'],\n                single_class_predictions=metrics[\'single_class_predictions\'])\n            csv_rows.append(row)\n            print(\'[evaluation]\',spec[\'name\'],seed,stage,split,json.dumps(metrics),flush=True)\n        if stage==\'final\':\n            last=json.loads((directory/\'monitor.jsonl\').read_text().splitlines()[-1])\n            assert last[\'final\'] and last[\'env_steps\']==env_steps\n            for key in [\'accuracy\',\'balanced_accuracy\',\'macro_f1\',\'avg_t\']:\n                assert np.isclose(last[key],record[\'validation\'][key]),(seed,key)\n        checks.append(dict(seed=seed,stage=stage,checkpoint_sha256_verified=True,\n            split_matches_reference=True,finite_parameters=True,parameters=parameters,\n            metrics_independently_recomputed=True,actual_optimizer_counter_verified=not static))\n        runs.append(record)\n        del model\n    main_stage=\'selected\' if static else \'final\'\n    aggregates={}\n    for stage in sorted({r[\'stage\'] for r in runs}):\n        aggregates[stage]={split:aggregate([r[split] for r in runs if r[\'stage\']==stage]) for split in [\'validation\',\'test\']}\n    final=dict(experiment=spec,seeds=[0,1,2],train_n=800,validation_n=200,test_n=6164,\n        split_seed=0,train_hash=sha(data_dir/\'wafer_TRAIN\'),test_hash=sha(data_dir/\'wafer_TEST\'),\n        actual_new_training=True,main_stage=main_stage,aggregate=aggregates[main_stage][\'test\'],\n        validation_aggregate=aggregates[main_stage][\'validation\'],stage_aggregates=aggregates,\n        runs=runs,independent_training_runs=3,TEST_used_for_selection=False,\n        official_TEST_examined_in_previous_project_rounds=True,\n        single_class_runs=sum(r[\'test\'][\'single_class_predictions\'] for r in runs if r[\'stage\']==main_stage))\n    write_json(ROOT/\'FINAL_RESULTS.json\',final)\n    with (ROOT/\'FINAL_TEST_RESULTS.csv\').open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as stream:\n        writer=csv.DictWriter(stream,fieldnames=list(csv_rows[0]));writer.writeheader();writer.writerows(csv_rows)\n    write_json(ROOT/\'VERIFICATION.json\',dict(all_passed=True,checks=checks,\n        checkpoint_locks_before_TEST=locks,data_hashes=dict(TRAIN=sha(data_dir/\'wafer_TRAIN\'),TEST=sha(data_dir/\'wafer_TEST\')),\n        source_hashes={name:sha(ROOT/name) for name in [\'ppo_wafer.py\',\'balanced_ppo.py\',\'equal_budget_ppo.py\',\'run_equal_budget_experiments.py\',\'evaluate_final.py\']},\n        execution=\'actual Python notebook cells with captured outputs and timings; no Jupyter kernel\',\n        every_evaluation_starts_at_t1=True,no_TEST_checkpoint_selection=True,\n        generated_utc=datetime.now(timezone.utc).isoformat()))\n    primary=[r for r in runs if r[\'stage\']==main_stage]\n    lines=[f"# {spec[\'name\']}", \'\',spec[\'description\'],\'\',\n        \'Three fresh independent trainings; seeds 0, 1, 2. Same original Wafer TRAIN split: 800 training / 200 validation, split_seed=0. Official TEST: 6,164 series; length 152.\', \'\',\n        \'## Final results\', \'\', \'| Split | Seed | Accuracy (%) | BA (%) | Macro-F1 (%) | Mean observations |\',\n        \'|---|---:|---:|---:|---:|---:|\']\n    for split in [\'validation\',\'test\']:\n        for r in primary:\n            m=r[split]; lines.append(f"| {split} | {r[\'seed\']} | {100*m[\'accuracy\']:.4f} | {100*m[\'balanced_accuracy\']:.4f} | {100*m[\'macro_f1\']:.4f} | {m[\'avg_t\']:.4f} |")\n        a=aggregates[main_stage][split]\n        vals=[f"{100*a[k][\'mean\']:.4f} ± {100*a[k][\'sample_sd\']:.4f}" for k in [\'accuracy\',\'balanced_accuracy\',\'macro_f1\']]\n        lines.append(f"| {split} | Mean ± SD | {\' | \'.join(vals)} | {a[\'avg_t\'][\'mean\']:.4f} ± {a[\'avg_t\'][\'sample_sd\']:.4f} |")\n    lines += [\'\', \'## Controlled configuration\', \'\', json.dumps(spec,indent=2), \'\',\n        \'PPO constants: natural class sampling, WAIT=-0.0001*t**(1/3), correct/incorrect=+1/-1, unweighted critic and environment rewards, LR=3e-4, gamma=1, GAE=.95, entropy=.02, value coefficient=.5, clip=.2, max gradient norm=.5, 32 environments x 64 steps, minibatch 256, two CPU threads per seed. Actor and critic hidden layers are 64 units each. All evaluations start at observation one and use greedy decisions.\', \'\',\n        \'Legacy training: 5 PPO epochs per rollout, target_kl=.03 and 1,048,576 interactions. This is the earlier training recipe, not a change to only one isolated hyperparameter. Actual Adam updates are counted. A fixed snapshot at exactly 100,000 interactions is also evaluated; it occurs during collection, following the completed rollout at 98,304 interactions, without adding an update.\', \'\',\n        \'Current training: exactly 100,000 interactions and 100,000 actual Adam steps, 256 epochs per full rollout, target_kl disabled. The final rollout is shortened as in the nine reference experiments. Old rollouts are discarded before new collection.\', \'\',\n        \'Current-only ablation: actor and critic receive only scalar x_t. No historical observations, explicit clock, trainable feature encoder, substitute extractor or recurrent memory. FlattenExtractor is an identity/reshape with zero parameters; actor and critic themselves remain (516 parameters in total). Random episode initialization stays unchanged.\', \'\',\n        \'Start-at-one ablation: only training episode initialization changes to t=1. The full observed-prefix CNN remains (95,428 total PPO parameters).\', \'\',\n        \'Static t=2 control: a supervised network receives only [x1,x2], padded with zeros to 152 for the same CNN and 64-unit actor trunk. Two class logits replace the action head; no critic or WAIT exists (87,042 parameters). Class-weighted cross-entropy uses alpha=.5, Adam lr=3e-4/eps=1e-5, batch 256, at most 256 epochs. Minimum weighted validation cross-entropy selects the checkpoint; early stopping patience is 30 epochs. Supervised minibatch updates are reported separately and are not called environment interactions.\', \'\',\n        \'## Files and reproduction\', \'\',\n        \'Exactly 15 top-level items: executed notebook, five Python source files, requirements, license, FINAL_RESULTS.json, FINAL_TEST_RESULTS.csv, README.md, PROVENANCE.json, VERIFICATION.json, data/ and models/. CSV contains BOTH validation and TEST, including the legacy 100k stage. JSON contains all metrics, per-seed values and means/sample SDs. Every seed stores weights, configs, split indices, counters, validation history and predictions for both splits.\', \'\',\n        \'The notebook embeds the exact source and data, and executes three fresh seed processes. Captured outputs, execution counts and timings record actual Python-cell execution; no Jupyter kernel is used. Use a new empty PROJECT for a fresh notebook rerun. Existing models are never resumed or silently overwritten.\', \'\',\n        \'Single-seed terminal equivalent: python run_equal_budget_experiments.py --seed 0 --output NEW_OUTPUT/seed0 --threads 2\',\n        \'Evaluation: python evaluate_final.py --models-root NEW_OUTPUT --data-dir data --threads 2\', \'\',\n        \'All five settings were declared before these results. TEST has been examined in earlier project rounds; it is not described as untouched. No TEST result chooses checkpoints or changes settings. Archived UCR normalization is retained, matching the nine references.\', \'\']\n    (ROOT/\'README.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n    assert len([p for p in ROOT.iterdir() if p.name!=\'__pycache__\'])==15, sorted(p.name for p in ROOT.iterdir())\n    print(\'VERIFIED FINAL AGGREGATES\',json.dumps(aggregates,indent=2),flush=True)\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--models-root\',default=\'models\');p.add_argument(\'--data-dir\',default=\'data\');p.add_argument(\'--threads\',type=int,default=2)\n    a=p.parse_args();main(Path(a.models_root),Path(a.data_dir),a.threads)\n',encoding='utf-8')
print('Exact source ready: evaluate_final.py')

Exact source ready: evaluate_final.py


In [8]:
DATA.mkdir(parents=True,exist_ok=True)
(DATA/'wafer_TRAIN').write_bytes(bytes.fromhex('...'))
(DATA/'wafer_TEST').write_bytes(bytes.fromhex('...'))
assert hashlib.sha256((DATA/'wafer_TRAIN').read_bytes()).hexdigest()=='31bba0c41bd068e5b9f3fc04ce7856dc23d93758d2e09cc53ca4a32d3b7a0f05'
assert hashlib.sha256((DATA/'wafer_TEST').read_bytes()).hexdigest()=='bfce80cbcb14d2623c64e70c81ef8a83204c79a5fa739cd247c78b5b34610605'
print('Original TRAIN and TEST bytes reconstructed and verified; training reads TRAIN only')


Original TRAIN and TEST bytes reconstructed and verified; training reads TRAIN only


In [9]:
import numpy as np
from sklearn.model_selection import train_test_split
from ppo_wafer import load_split
from run_equal_budget_experiments import preflight
X,y=load_split(DATA,'TRAIN')
train_ids,val_ids=train_test_split(np.arange(len(y)),test_size=.2,stratify=y,random_state=0)
preflight(X,y,train_ids,val_ids)
print('Frozen experiment',NAME,'seeds',SEEDS,'same 800/200 split confirmed',flush=True)


[preflight] exact split, current-only absence of past/future, WAIT mask, start-at-one and two-point input passed
Frozen experiment ppo_alpha05_start1 seeds [0, 1, 2] same 800/200 split confirmed


In [10]:
if RUN_TRAINING:
    TRAINING_OUTPUT.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=NAME+'_stdout_') as log_directory:
        processes={};handles={}
        try:
            for seed in SEEDS:
                output=TRAINING_OUTPUT/f'seed{seed}'
                output.mkdir(parents=True,exist_ok=True)
                handle=(Path(log_directory)/f'seed{seed}.log').open('w',buffering=1)
                command=[sys.executable,'-u',str(PROJECT/'run_equal_budget_experiments.py'),'--seed',str(seed),'--output',str(output),'--threads',str(THREADS)]
                environment=os.environ.copy()
                environment.update(OMP_NUM_THREADS='2',MKL_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',PYTHONUNBUFFERED='1')
                processes[seed]=subprocess.Popen(command,stdout=handle,stderr=subprocess.STDOUT,env=environment)
                handles[seed]=handle
                print('STARTED FRESH TRAINING',NAME,'seed',seed,'pid',processes[seed].pid,flush=True)
            while True:
                statuses={s:p.poll() for s,p in processes.items()}
                failed={s:status for s,status in statuses.items() if status not in (None,0)}
                if failed:
                    for seed in failed:print((Path(log_directory)/f'seed{seed}.log').read_text(),flush=True)
                    raise RuntimeError(f'Training failed: {failed}')
                report=[]
                for seed,status in statuses.items():
                    monitor=TRAINING_OUTPUT/f'seed{seed}'/'monitor.jsonl'
                    rows=[json.loads(l) for l in monitor.read_text().splitlines() if l.strip()] if monitor.exists() else []
                    last=rows[-1] if rows else {}
                    report.append(dict(seed=seed,exit_status=status,environment_steps=last.get('env_steps'),actual_updates=last.get('actual_gradient_steps'),epoch=last.get('epoch'),validation_accuracy=last.get('accuracy'),validation_macro_f1=last.get('macro_f1'),mean_observations=last.get('avg_t')))
                print(datetime.datetime.now(datetime.timezone.utc).isoformat(),json.dumps(report),flush=True)
                if all(status==0 for status in statuses.values()):break
                time.sleep(30)
            for seed in SEEDS:
                summary=json.loads((TRAINING_OUTPUT/f'seed{seed}'/'training_summary.json').read_text())
                assert summary['completed']
                handles[seed].flush()
                print('ACTUAL TRAINING STDOUT',NAME,'SEED',seed,flush=True)
                print((Path(log_directory)/f'seed{seed}.log').read_text(),flush=True)
            print('ALL THREE FRESH TRAININGS COMPLETED',NAME,flush=True)
        finally:
            for seed,process in processes.items():
                if process.poll() is None:process.terminate()
                handles[seed].close()


STARTED FRESH TRAINING ppo_alpha05_start1 seed 0 pid 280
STARTED FRESH TRAINING ppo_alpha05_start1 seed 1 pid 281
STARTED FRESH TRAINING ppo_alpha05_start1 seed 2 pid 282
2026-11-17T13:41:38.366955+00:00 [{"seed": 0, "exit_status": null, "environment_steps": null, "actual_updates": null, "epoch": null, "validation_accuracy": null, "validation_macro_f1": null, "mean_observations": null}, {"seed": 1, "exit_status": null, "environment_steps": null, "actual_updates": null, "epoch": null, "validation_accuracy": null, "validation_macro_f1": null, "mean_observations": null}, {"seed": 2, "exit_status": null, "environment_steps": null, "actual_updates": null, "epoch": null, "validation_accuracy": null, "validation_macro_f1": null, "mean_observations": null}]
2026-11-17T13:42:08.370934+00:00 [{"seed": 0, "exit_status": null, "environment_steps": null, "actual_updates": null, "epoch": null, "validation_accuracy": null, "validation_macro_f1": null, "mean_observations": null}, {"seed": 1, "exit_sta

In [11]:
if RUN_FINAL_TEST:
    evaluated=subprocess.run([sys.executable,'-u',str(PROJECT/'evaluate_final.py'),'--models-root',str(TRAINING_OUTPUT),'--data-dir',str(DATA),'--threads',str(THREADS)],capture_output=True,text=True)
    print(evaluated.stdout,end='')
    if evaluated.stderr:print(evaluated.stderr,file=sys.stderr,end='')
    evaluated.check_returncode()
    final=json.loads((PROJECT/'FINAL_RESULTS.json').read_text())
    assert final['actual_new_training'] and final['independent_training_runs']==3
    print('ACTUAL VALIDATION AND TEST COMPLETE',NAME,flush=True)


[evaluation] ppo_alpha05_start1 0 final validation {"n": 200, "evaluation": "greedy", "accuracy": 0.995, "balanced_accuracy": 0.9736842105263157, "macro_f1": 0.9851090760181669, "weighted_f1": 0.994939319484774, "minority_f1": 0.972972972972973, "minority_precision": 1.0, "minority_recall": 0.9473684210526315, "majority_recall": 1.0, "precision_by_class": [1.0, 0.9945054945054945], "f1_by_class": [0.972972972972973, 0.9972451790633609], "recall_by_class": [0.9473684210526315, 1.0], "mcc": 0.9706508641410859, "cohen_kappa": 0.9702203692674211, "confusion_matrix": [[18, 1], [0, 181]], "avg_t": 1.4, "earliness": 0.009210526315789473, "first_stop_fraction": 0.7, "mean_return": 0.9899574007895011, "avg_t_by_class": [2.0526315789473686, 1.3314917127071824], "decision_time_median": 1.0, "decision_time_p90": 2.0999999999999943, "decision_time_p95": 3.0, "decision_time_max": 3, "predicted_class_counts": [18, 182], "single_class_predictions": false, "action_counts": [80, 18, 182], "first_probs_b